In [16]:
# imports & config
import os
import math
import torch
from PIL import Image, ImageFilter, ImageDraw
import torchvision.transforms as T
import numpy as np

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", DEVICE)

# Directories 
TEST_IMAGES = "test_images"
LABEL_DIR = "yolo_detect_output/labels"
OUTPUT_DIR = "output_mosaic"
STYLE_MODEL_PATH = "saved_models/mosaic.pth"  

os.makedirs(OUTPUT_DIR, exist_ok=True)


Using device: cpu


In [17]:
# Load TransformerNet
from transformer_net import TransformerNet

def load_transformer(checkpoint):
    model = TransformerNet()
    state = torch.load(checkpoint, map_location=DEVICE)

    # Fix old state dict format
    for k in list(state.keys()):
        if "running_" in k:
            del state[k]

    model.load_state_dict(state)
    model.to(DEVICE).eval()
    return model

style_model = load_transformer(STYLE_MODEL_PATH)
print("Style model loaded:", STYLE_MODEL_PATH)


Style model loaded: saved_models/mosaic.pth


C:\Users\anany\AppData\Local\Temp\ipykernel_21372\2312837370.py:6: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  state = torch.load(checkpoint, map_location=DEVICE)


In [18]:
# Helpers
to_tensor = T.ToTensor()
to_pil = T.ToPILImage()

def make_div4(n):
    return int(math.ceil(n / 4) * 4)

def prepare_patch(patch, max_dim=512):
    w, h = patch.size

    # keep aspect ratio
    if max(w, h) > max_dim:
        if w >= h:
            new_w = max_dim
            new_h = int(h * max_dim / w)
        else:
            new_h = max_dim
            new_w = int(w * max_dim / h)
        patch = patch.resize((new_w, new_h), Image.LANCZOS)

    # fit to network: divisible by 4
    nw, nh = patch.size
    nw = make_div4(nw)
    nh = make_div4(nh)
    patch = patch.resize((nw, nh), Image.LANCZOS)

    return patch

def apply_style(patch):
    patch = prepare_patch(patch)

    # 1) To tensor in [0,1]  -> scale to [0,255] for the style model
    t = to_tensor(patch).unsqueeze(0) * 255.0
    t = t.to(DEVICE)

    with torch.no_grad():
        out = style_model(t).cpu().squeeze(0)   # still in [0,255]

    # 2) Back to [0,1] for PIL
    out = out / 255.0
    out = out.clamp(0.0, 1.0)

    return to_pil(out)


def soft_mask(size, blur=22):
    w, h = size
    mask = Image.new("L", (w, h), 0)
    draw = ImageDraw.Draw(mask)
    draw.rectangle((blur, blur, w-blur, h-blur), fill=255)
    return mask.filter(ImageFilter.GaussianBlur(blur))


In [19]:
# Helper to read YOLO txt file
def load_yolo_bbox(txt_file, img_w, img_h):
    boxes = []
    with open(txt_file, "r") as f:
        for line in f:
            numbers = list(map(float, line.split()))
            if len(numbers) < 5:
                continue  # skip malformed lines
            c, x_c, y_c, w_n, h_n = numbers[:5]  # ignore extra values
            cx = x_c * img_w
            cy = y_c * img_h
            w = w_n * img_w
            h = h_n * img_h
            x1 = int(cx - w/2)
            y1 = int(cy - h/2)
            x2 = int(cx + w/2)
            y2 = int(cy + h/2)
            boxes.append((x1, y1, x2, y2))
    return boxes



In [20]:
# Process all test images
images = [f for f in os.listdir(TEST_IMAGES) if f.endswith((".jpg",".png",".jpeg"))]
print("Found test images:", len(images))

for name in images:
    img_path = os.path.join(TEST_IMAGES, name)
    txt_path = os.path.join(LABEL_DIR, name.replace(".jpg", ".txt").replace(".png", ".txt"))

    if not os.path.exists(txt_path):
        print("Skipping (no labels):", name)
        continue

    img = Image.open(img_path).convert("RGB")
    W, H = img.size

    # load all (usually 1) wineglass boxes
    boxes = load_yolo_bbox(txt_path, W, H)
    out = img.copy()

    for (x1, y1, x2, y2) in boxes:
        patch = img.crop((x1, y1, x2, y2))
        styled = apply_style(patch)

        # resize styled patch to exact box shape
        styled = styled.resize((x2-x1, y2-y1), Image.LANCZOS)

        # mask for smooth edges
        mask = soft_mask((x2-x1, y2-y1))

        # paste
        out.paste(styled, (x1, y1), mask)

    save_path = os.path.join(OUTPUT_DIR, name)
    out.save(save_path)
    print("Saved:", save_path)


Found test images: 22
Saved: output_mosaic\005_jpg.rf.122abecdeedab52c32e140f52a2c049e.jpg
Saved: output_mosaic\027_jpg.rf.688eafba8895528950aaf642f268b900.jpg
Saved: output_mosaic\029_jpg.rf.d1e81a9cfdd429119b560dec374381b8.jpg
Saved: output_mosaic\031_jpg.rf.80bd853884dc58d6a8fc13704f9cf584.jpg
Saved: output_mosaic\044_jpg.rf.ec5c24222e61dc5674b6e04e996dbab4.jpg
Saved: output_mosaic\047_jpg.rf.2f3188b4f56b0900db729e2c85461b41.jpg
Saved: output_mosaic\048_jpg.rf.d17c03fb2495348eec2847ac45d4702b.jpg
Saved: output_mosaic\049_jpg.rf.418f5a2b8bd44fff16fe5b509a40c068.jpg
Saved: output_mosaic\071_jpg.rf.8dea64c8151dc1561fad0ef4413ba48d.jpg
Saved: output_mosaic\092_jpg.rf.aa0e35fc40690d291317d76b7e967d46.jpg
Saved: output_mosaic\103_jpg.rf.ae981f8a55bb9fa2403259c42c8e6106.jpg
Saved: output_mosaic\112_jpg.rf.5297b0556810ed89552aa0ee9aed32ee.jpg
Saved: output_mosaic\122_jpg.rf.ba8054ccd52d8bebecc857551c516d7a.jpg
Saved: output_mosaic\125_jpg.rf.8efe43215efab3079185f06b4e7c29f7.jpg
Saved: outpu